In [1]:
!pip install -U sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.2/470.2 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 52.2 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstallin

In [4]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

# 모델 로드
model = SentenceTransformer('snunlp/KR-SBERT-V40K-klueNLI-augSTS')

# 데이터 로드
file_path = '/content/drive/MyDrive/sim/cleaned_gpt_generated_expression.csv'
df = pd.read_csv(file_path)
df.fillna("", inplace=True)

from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 문장 리스트 생성
gen_sentences = df["generated_sentence"].tolist()
meanings_all = df[["meaning1", "meaning2", "meaning3", "meaning4"]].values.tolist()

# generated_sentence 배치 인코딩
gen_embeddings = model.encode(gen_sentences, batch_size=32, show_progress_bar=True)

# 의미 표현 flatten 후 배치 인코딩
flat_meanings = [m for sublist in meanings_all for m in sublist]
meaning_embeddings = model.encode(flat_meanings, batch_size=32, show_progress_bar=True)

# 유사도 계산
similarities = []
for i in range(len(df)):
    gen_vec = gen_embeddings[i]
    mean_vecs = meaning_embeddings[i*4:(i+1)*4]

    sims = [
        cosine_similarity([gen_vec], [mv])[0][0]
        for mv in mean_vecs
        if mv is not None and np.any(mv)
    ]

    similarities.append(max(sims) if sims else 0.0)

df["sbert"] = similarities

output_path = '/content/drive/MyDrive/sim/cleaned_gpt_generated_with_sbert_fast.csv'
df.to_csv(output_path, index=False)
print(f"✅ 저장 완료: {output_path}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Batches:   0%|          | 0/317 [00:00<?, ?it/s]

Batches:   0%|          | 0/1265 [00:00<?, ?it/s]

✅ 저장 완료: /content/drive/MyDrive/sim/cleaned_gpt_generated_with_sbert_fast.csv
